# 03 · Baselines: how far do simple rules get?

Before any machine learning, we measure **rules that need no training**. If a model can't beat them, it isn't worth deploying.

### Metrics (how the benchmark scores RCA)
For each case, sort services by score. Find the position (rank) of the true root cause.
- **AC@1** = share of cases where the true cause is ranked **first** (top-1 accuracy)
- **AC@3 / AC@5** = share where it is in the top 3 / top 5
- **Avg@5** = average of AC@1…AC@5 (the RCAEval summary number)
- **MRR** = average of 1 / rank

Ties are counted *pessimistically* (if the culprit shares the top score with 4 others, it counts as rank 5), so a rule cannot look good by luck.

### The rules
| Rule | Score of a service |
|---|---|
| random | random number (expected value shown) |
| **max metric z (BARO-style)** | its largest robust z-score over all its metrics |
| latency only | largest z of latency-50/90 |
| resource only | largest z of cpu / mem / diskio / socket |
| log errors | increase in error-like log lines |
| trace p95 | increase in p95 span latency (OB, TT only) |
| rank average | average of its within-case ranks over all features |

These rules use **all 270 cases** (no training, so no split needed).

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*(mismatched devices|pandas version 4).*")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
RESULTS = ROOT / "reports" / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

from rcalens.data import load_features
from rcalens.evaluate import evaluate, true_ranks, rank_metrics
df, has_traces = load_features()
rng = np.random.default_rng(0)

def with_score(values):
    d = df[["case", "system", "service", "is_rc"]].copy(); d["score"] = np.nan_to_num(np.asarray(values, float), nan=-1e9); return d

z = lambda cols: df[[c for c in cols if c in df]].max(axis=1)
rules = {
    "max metric z (BARO-style)": df["m_all_maxz"],
    "latency only": z(["m_latency50_maxz", "m_latency90_maxz"]),
    "resource only (cpu/mem/disk/socket)": z(["m_cpu_maxz", "m_mem_maxz", "m_diskio_maxz", "m_socket_maxz"]),
    "log errors": df.get("l_err_ratio", pd.Series(np.nan, index=df.index)),
    "rank average (all features)": df[[c for c in df.columns if c.endswith("_rank")]].mean(axis=1),
}
if has_traces:
    rules["trace p95"] = df["t_p95_ratio"]

# random: expected metrics computed exactly from the number of candidates
n_cand = df.groupby("case").size()
exp_rand = {f"AC@{k}": float(np.mean(np.minimum(k, n_cand) / n_cand)) for k in (1, 3, 5)}
exp_rand["Avg@5"] = float(np.mean([np.mean(np.minimum(k, n_cand) / n_cand) for k in range(1, 6)]))
exp_rand["MRR"] = float(np.mean([(1 / np.arange(1, n + 1)).mean() for n in n_cand]))

tables = {"random (expected)": pd.Series(exp_rand)}
per_system = {}
for name, s in rules.items():
    ev = evaluate(with_score(s)); tables[name] = ev.loc["all"]; per_system[name] = ev
base = pd.DataFrame(tables).T
base.round(3)

**Reading it:** compare each rule with *random*. The strongest single rule is the one to beat in notebook 04.

### Per system (AC@1, top-1 accuracy)

In [ ]:
pd.DataFrame({k: v["AC@1"] for k, v in per_system.items()}).T.round(3)

### Per fault type (AC@1 of the best rule)
Which faults are easy and which are hard for the rule?

In [ ]:
best_rule = base.drop(index="random (expected)")["AC@1"].idxmax()
r = true_ranks(with_score(rules[best_rule]))
fault = df.drop_duplicates("case").set_index("case")["fault"]
print("best rule:", best_rule)
(r <= 1).groupby(fault.reindex(r.index).to_numpy()).mean().round(3).rename("AC@1").to_frame().T

In [ ]:
json.dump({"has_traces": has_traces, "best_rule": best_rule, "overall": base.round(4).to_dict(orient="index"),
           "ac1_by_system": pd.DataFrame({k: v["AC@1"] for k, v in per_system.items()}).round(4).to_dict()},
          open(RESULTS / "baselines.json", "w"), indent=1)
with_score(rules[best_rule]).to_parquet(ROOT / "data" / "processed" / "baseline_scores.parquet", index=False)
print("saved reports/results/baselines.json")

### Takeaways
- A single robust statistic (the BARO idea) is already far above random: injected faults usually leave a clear mark on the culprit's own metrics.
- It struggles where the symptom *travels*: delay and loss faults make callers slow too, so the slowest service is often not the cause.
- That's the opening for a learned model that combines many weak signals (resource vs latency, logs, traces, ranks).

➡️ Next: **04_ml_ranker**: learning to rank with XGBoost.